Keras model for running inference. Copied our baseline and a HGQ-model

In [ ]:
keras_model_path = "model_Training_AdaptiveHP_acc=0.9578_ebops=34304.keras"
#keras_model_path = "MNIST_baseline_model.keras"
#keras_model_path = "mnist_model.keras"

import numpy as np
import keras

from keras.models import load_model
#from qkeras.utils import load_qmodel
import hgq.layers

model = load_model(keras_model_path)

from hgq.utils import trace_minmax
#trace_minmax(model, X_test, verbose=True)

Process an image
[PIL modes](https://pillow.readthedocs.io/en/stable/handbook/concepts.html).
MNIST is trained on 28x28 grayscale (1 channel) uint8 single-digits, 0-9, white on black [ref](https://www.tensorflow.org/datasets/catalog/mnist).

Based on 
- [JOBIN456/MNIST-RealTime-OpenCV-Keras](https://github.com/JOBIN456/MNIST-RealTime-OpenCV-Keras) 
- [stackoverflow.com/questions/52068277/change-frame-rate-in-opencv-3-4-2](https://stackoverflow.com/questions/52068277/change-frame-rate-in-opencv-3-4-2)
- GPT-5.2 Codex for many of the options. Needed to test a lot of different combinations.

<div>
<img src="https://storage.googleapis.com/tfds-data/visualization/fig/mnist-3.0.1.png" width="500"/>
</div>

In [7]:
import ipywidgets as widgets
from IPython.display import display
import cv2

mirror_toggle = widgets.Checkbox(
    value=False, description="Mirror image"
 )
threshold_slider = widgets.IntSlider(
    value=120, min=0, max=255, step=1, description="Threshold"
 )

interp_options = [
    ("INTER_NEAREST", cv2.INTER_NEAREST),
    ("INTER_LINEAR", cv2.INTER_LINEAR),
    ("INTER_CUBIC", cv2.INTER_CUBIC),
    ("INTER_AREA", cv2.INTER_AREA),
    ("INTER_LANCZOS4", cv2.INTER_LANCZOS4),
    ("INTER_LINEAR_EXACT", cv2.INTER_LINEAR_EXACT),
    ("INTER_NEAREST_EXACT", cv2.INTER_NEAREST_EXACT),
 ]

resize_interp_dropdown = widgets.Dropdown(
    options=interp_options,
    value=cv2.INTER_NEAREST,
    description="Resize interp"
 )
display_interp_dropdown = widgets.Dropdown(
    options=interp_options,
    value=cv2.INTER_NEAREST,
    description="Display interp"
 )

use_roi_toggle = widgets.Checkbox(
    value=True, description="Crop before processing (ROI)"
 )
roi_size_slider = widgets.IntSlider(
    value=240, min=120, max=480, step=20, description="Crop size"
 )
threshold_mode_dropdown = widgets.Dropdown(
    options=["none", "fixed", "otsu", "adaptive"],
    value="fixed",
    description="Thresh mode"
 )
invert_mode_dropdown = widgets.Dropdown(
    options=["auto", "force_white", "force_black"],
    value="auto",
    description="Invert mode"
 )
blur_kernel_slider = widgets.IntSlider(
    value=1, min=1, max=9, step=2, description="Blur k"
 )
morph_open_toggle = widgets.Checkbox(
    value=False, description="Morph open"
 )
morph_iter_slider = widgets.IntSlider(
    value=1, min=1, max=3, step=1, description="Morph iters"
 )
center_scale_toggle = widgets.Checkbox(
    value=False, description="Center+scale"
 )
norm_mode_dropdown = widgets.Dropdown(
    options=["0-to-1", "mean_std"],
    value="0-to-1",
    description="Normalize"
 )

display(
    mirror_toggle,
    threshold_slider,
    threshold_mode_dropdown,
    invert_mode_dropdown,
    blur_kernel_slider,
    morph_open_toggle,
    morph_iter_slider,
    center_scale_toggle,
    use_roi_toggle,
    roi_size_slider,
    resize_interp_dropdown,
    display_interp_dropdown,
    norm_mode_dropdown
 )

Checkbox(value=False, description='Mirror image')

IntSlider(value=120, description='Threshold', max=255)

Dropdown(description='Thresh mode', index=1, options=('none', 'fixed', 'otsu', 'adaptive'), value='fixed')

Dropdown(description='Invert mode', options=('auto', 'force_white', 'force_black'), value='auto')

IntSlider(value=1, description='Blur k', max=9, min=1, step=2)

...

Checkbox(value=True, description='Crop before processing (ROI)')

IntSlider(value=240, description='Crop size', max=480, min=120, step=20)

Dropdown(description='Resize interp', options=(('INTER_NEAREST', 0), ('INTER_LINEAR', 1), ('INTER_CUBIC', 2), …

Dropdown(description='Display interp', options=(('INTER_NEAREST', 0), ('INTER_LINEAR', 1), ('INTER_CUBIC', 2),…

Dropdown(description='Normalize', options=('0-to-1', 'mean_std'), value='0-to-1')

In [8]:
import cv2
import numpy as np
import time

cap = cv2.VideoCapture(0)
frame_rate = 300
prev = 0

while True:
    time_elapsed = time.time() - prev
    ret, frame = cap.read()
    if not ret:
        break
    if mirror_toggle.value:
        frame = cv2.flip(frame, 1)

    if time_elapsed > 1./frame_rate:
        prev = time.time()
        # Convert full frame to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Optional center crop ROI
        if use_roi_toggle.value:
            h, w = gray.shape
            roi_size = int(roi_size_slider.value)
            roi_size = max(20, min(roi_size, h, w))
            x0 = w // 2 - roi_size // 2
            y0 = h // 2 - roi_size // 2
            roi = gray[y0:y0 + roi_size, x0:x0 + roi_size]
        else:
            roi = gray

        # Optional blur to reduce noise
        blur_k = int(blur_kernel_slider.value)
        if blur_k > 1:
            roi = cv2.GaussianBlur(roi, (blur_k, blur_k), 0)

        # Thresholding (optional)
        threshold_value = int(threshold_slider.value)
        thresh_mode = threshold_mode_dropdown.value
        if thresh_mode == "otsu":
            _, processed = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif thresh_mode == "adaptive":
            processed = cv2.adaptiveThreshold(
                roi, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, 11, 2
            )
        elif thresh_mode == "fixed":
            _, processed = cv2.threshold(roi, threshold_value, 255, cv2.THRESH_BINARY_INV)
        else:
            processed = roi

        # Invert only when needed (binary only)
        if processed.dtype == np.uint8 and processed.ndim == 2 and thresh_mode != "none":
            invert_mode = invert_mode_dropdown.value
            if invert_mode == "force_white":
                processed = cv2.bitwise_not(processed)
            elif invert_mode == "auto":
                if np.mean(processed) > 127:
                    processed = cv2.bitwise_not(processed)

        # Morphology open to remove speckles (binary only)
        if morph_open_toggle.value and thresh_mode != "none":
            kernel = np.ones((3, 3), np.uint8)
            processed = cv2.morphologyEx(
                processed, cv2.MORPH_OPEN, kernel, iterations=int(morph_iter_slider.value)
            )

        # Center and scale using largest contour (binary only)
        if center_scale_toggle.value and thresh_mode != "none":
            contours, _ = cv2.findContours(processed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                x, y, bw, bh = cv2.boundingRect(c)
                digit = processed[y:y + bh, x:x + bw]
                if digit.size > 0:
                    size = max(bw, bh)
                    square = np.zeros((size, size), dtype=np.uint8)
                    x_off = (size - bw) // 2
                    y_off = (size - bh) // 2
                    square[y_off:y_off + bh, x_off:x_off + bw] = digit
                    processed = square

        # Resize to 28x28
        resized = cv2.resize(
            processed, (28, 28), interpolation=int(resize_interp_dropdown.value)
        )

        # Normalize
        norm_mode = norm_mode_dropdown.value
        if norm_mode == "mean_std":
            resized = resized.astype("float32")
            resized = (resized - 0.1307 * 255.0) / (0.3081 * 255.0)
        else:
            resized = resized.astype("float32") / 255.0

        # Reshape to (1,28,28,1)
        input_img = resized.reshape(1, 28, 28, 1)

        # Predict
        prediction = model.predict(input_img, verbose=0)
        digit = np.argmax(prediction)
        confidence = np.max(prediction)
        top3_idx = np.argsort(prediction[0])[::-1][:3]
        top3 = [(int(i), float(prediction[0][i])) for i in top3_idx]

        # Display the 28x28 input scaled up with prediction
        display_img = cv2.resize(
            resized, (480, 480), interpolation=int(display_interp_dropdown.value)
        )
        display_img = display_img.copy()
        if display_img.dtype != np.uint8:
            if norm_mode == "mean_std":
                display_img = np.clip((display_img + 1.0) * 127.5, 0, 255).astype(np.uint8)
            else:
                display_img = np.clip(display_img * 255.0, 0, 255).astype(np.uint8)
        cv2.imshow("Model input", display_img)

    if use_roi_toggle.value:
        h, w = frame.shape[:2]
        roi_size = int(roi_size_slider.value)
        roi_size = max(20, min(roi_size, h, w))
        x0 = w // 2 - roi_size // 2
        y0 = h // 2 - roi_size // 2
        cv2.rectangle(frame, (x0, y0), (x0 + roi_size, y0 + roi_size), (0, 255, 0), 2)

    text_color = (50, 50, 220)
    cv2.putText(frame, f"Prediction: {digit} ({confidence:.2f})",
        (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, text_color, 2)
    if "top3" in locals():
        cv2.putText(frame, f"Next: {top3[1][0]} ({top3[1][1]:.2f})",
            (50, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, text_color, 2)
        cv2.putText(frame, f"Next: {top3[2][0]} ({top3[2][1]:.2f})",
            (50, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.8, text_color, 2)

    cv2.imshow("Live camera - Quit by hitting Q", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

2026-05-26 22:54:06.672956: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-05-26 22:54:07.155925: I external/local_xla/xla/service/service.cc:163] XLA service 0x7ccb6c0a7380 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-05-26 22:54:07.155939: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-05-26 22:54:07.226389: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779828847.653148  653435 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-05-26 22:54:07.848201: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-05-26 22:54:08.076088: I ten

In [4]:

cap.release()
cv2.destroyAllWindows()

Simple test of webcam feed

In [ ]:
import cv2
import numpy as np

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert full frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Resize full frame to 28x28
    resized = cv2.resize(gray, (28, 28))

    # Threshold for cleaner digit (better than simple invert)
    _, resized = cv2.threshold(resized, 120, 255, cv2.THRESH_BINARY_INV)

    # Normalize
    resized = resized.astype("float32") / 255.0


    cv2.imshow("Live feed", frame)
    cv2.imshow("Processed feed", resized)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()